# 第74课 · 让每个错词供出线索——ASR 错误分析（error analysis），从 WER 数字到修复路线

> **定位**：本课用代码统计和错误模式表，把 WER 拆成可执行的改进列表。

**学习目标**
- 从对齐（alignment）路径中提取 S/D/I 错误的具体词
- 统计最高频的错误对（substitution pairs）
- 识别哪类词对 ASR 最难（数字、专名、方言）


← **上一课**　[L73 · WER 评估](L73_wer_eval.ipynb)

> 上节课学习了 **WER 评估**：词错误率（插入 / 删除 / 替换），jiwer 对比逐句分析。  
> 本课将探讨 **ASR 错误分析**。

In [ ]:
import numpy as np
from collections import Counter

In [ ]:
# 复用 L67 的 alignment 函数
def alignment(ref_str, hyp_str):
    """返回词级别对齐操作列表：(op, ref_word, hyp_word)。op ∈ {M,S,I,D}"""
    # 统一小写后再比较，避免大小写差异影响错误分析口径
    a, b = ref_str.lower().split(), hyp_str.lower().split()
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(m+1): dp[i][0] = i
    for j in range(n+1): dp[0][j] = j
    for i in range(1,m+1):
        for j in range(1,n+1):
            if a[i-1]==b[j-1]: dp[i][j]=dp[i-1][j-1]
            else: dp[i][j]=1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    ops,i,j=[],m,n
    while i>0 or j>0:
        if i>0 and j>0 and a[i-1]==b[j-1]:
            ops.append(('M',a[i-1],b[j-1])); i-=1; j-=1
        elif i>0 and j>0 and dp[i][j]==1+dp[i-1][j-1]:
            ops.append(('S',a[i-1],b[j-1])); i-=1; j-=1
        elif j>0 and dp[i][j]==1+dp[i][j-1]:
            ops.append(('I','-',b[j-1])); j-=1
        else:
            ops.append(('D',a[i-1],'-')); i-=1
    return list(reversed(ops))


In [ ]:
def wer(ref_str, hyp_str):
    """词错率 = (S+D+I) / |ref|，用 alignment() 统计各操作"""
    # ✏️ TODO: ops = alignment(ref_str, hyp_str)
    # ✏️ TODO: 统计 S、D、I 的数量
    # ✏️ TODO: 返回 (S+D+I) / len(ref_str.split())
    raise NotImplementedError('请实现 wer')


In [ ]:
# 验证 WER 实现
try:
    result = wer("the cat sat", "the cat sat")
    assert result is not None, "⬜ wer 未实现"
    assert abs(result - 0.0) < 1e-9, f"相同句子 WER 应为 0，得到 {result}"

    result2 = wer("the cat sat", "the dog sat")
    assert abs(result2 - 1/3) < 0.01, f"一词不同 WER 应约为 0.33，得到 {result2}"

    print(f"✅ WER 验证通过（相同={result:.3f}，一词不同={result2:.3f}）")
except (NotImplementedError, TypeError):
    print("⬜ wer 未实现")

## 10 对 (ref, hyp) 样本

In [ ]:
PAIRS = [
    ("the quick brown fox jumps over the lazy dog",
     "the quick brown fox jump over the lazy dog"),
    ("she sells seashells by the seashore",
     "she cells seashells by the seashore"),
    ("call nine one one immediately",
     "call 911 immediately"),
    ("the meeting is scheduled for three pm on friday",
     "the meeting is scheduled for 3 pm on friday"),
    ("artificial intelligence is transforming healthcare",
     "artificial intelligence is transforming health care"),
    ("please send the report to john smith",
     "please send the report to john's myth"),
    ("the temperature is twenty three degrees celsius",
     "the temperature is 23 degrees celsius"),
    ("i would like a cup of coffee please",
     "i would like a cup of coffee"),
    ("deep learning models require large amounts of training data",
     "deep learning models require large amounts of training data"),
    ("natural language processing has many applications",
     "natural language process has many application"),
]

print(f'{"#":>2}  {"WER":>5}  REF')
print('-' * 55)
try:
    for i, (ref, hyp) in enumerate(PAIRS):
        w = wer(ref, hyp)
        print(f'{i:>2}  {w:>5.1%}  {ref[:45]}')
except (NotImplementedError, TypeError):
    print('⬜ wer 未实现 — 完成上方 TODO 后重新运行本格')

## 错误统计

In [ ]:
sub_counter = Counter()
del_counter = Counter()
ins_counter = Counter()

for ref, hyp in PAIRS:
    for op, rw, hw in alignment(ref, hyp):
        if op == 'S':
            sub_counter[(rw, hw)] += 1
        elif op == 'D':
            del_counter[rw] += 1
        elif op == 'I':
            ins_counter[hw] += 1

print('=== 最常见替换 (ref→hyp) ===')
for (r, h), cnt in sub_counter.most_common(8):
    print(f'  {r:15s} → {h:15s}  ×{cnt}')

print()
print('=== 最常见删除 ===')
for word, cnt in del_counter.most_common(5):
    print(f'  deleted: {word:15s}  ×{cnt}')

print()
print('=== 最常见插入 ===')
for word, cnt in ins_counter.most_common(5):
    print(f'  inserted: {word:15s}  ×{cnt}')

## 错误模式总结

| 错误类型 | 例子 | 根本原因 | 改进方向 |
|---|---|---|---|
| 数字/文字混淆 | nine one one → 911 | 训练数据格式不统一 | 后处理规范化 |
| 形近词替换 | sells → cells | 声学特征相似 | 更多上下文 / LM |
| 专有名词 | john smith → john's myth | OOV（out-of-vocabulary，词表外）词汇 | 热词增强（hotword boosting） / 微调 |
| 尾词删除 | coffee please → coffee | 句末能量低 | VAD 优化 |

**下一步 L75**：把这些分析做成可视化仪表板。

## 量化指标：如何用 S/D/I 定向改进

**WER 分解公式**：`WER = (S + D + I) / N_ref`

| 错误主导类型 | 含义 | 优先改进方向 |
|---|---|---|
| S（替换）主导 | 声学相似词混淆 | 更大声学模型 / 领域 LM |
| D（删除）主导 | 句末/弱读音漏识别 | VAD 调优 / 增强训练数据 |
| I（插入）主导 | 噪声被识别为词 | 降噪预处理 / 阈值调整 |

错误分析的价值：把 WER 从黑盒数字变成可操作的改进列表。

In [ ]:
# 验证：S/D/I 统计之和 = 总编辑操作数，WER 分解自洽
total_S = sum(sub_counter.values())
total_D = sum(del_counter.values())
total_I = sum(ins_counter.values())
total_ref_words = sum(len(ref.split()) for ref, _ in PAIRS)

wer_val = (total_S + total_D + total_I) / total_ref_words
print(f'替换（S）: {total_S}')
print(f'删除（D）: {total_D}')
print(f'插入（I）: {total_I}')
print(f'参考词总数（N_ref）: {total_ref_words}')
print(f'WER = ({total_S}+{total_D}+{total_I}) / {total_ref_words} = {wer_val:.3f}')

assert total_S + total_D + total_I > 0, '没有检测到任何错误'
assert 0 < wer_val < 1, f'WER={wer_val:.3f} 超出合理范围'
print('✅ S/D/I 统计自洽，WER 分解正确')

In [ ]:
# Q4: S/D/I 数值验证
try:
    wer_val = wer('the cat sat on the mat', 'the cat sits on mat')
    ops = alignment('the cat sat on the mat', 'the cat sits on mat')
    S = sum(1 for op,*_ in ops if op=='S')
    D = sum(1 for op,*_ in ops if op=='D')
    I = sum(1 for op,*_ in ops if op=='I')
    ref_len = len('the cat sat on the mat'.split())
    expected = (S + D + I) / ref_len
    assert abs(wer_val - expected) < 1e-9, f'WER 与 S/D/I 统计不一致: {wer_val:.3f} vs {expected:.3f}'
    print(f'✅ S/D/I 验证通过  S={S} D={D} I={I}  WER={wer_val:.3f}')
except (NotImplementedError, TypeError):
    print('⬜ wer 未实现')


## 本课收束

现在能把 WER 分解成 S（替换）/ D（删除）/ I（插入）三类错误，并通过词频计数把每类错误溯源到具体词汇和可改进方向（声学模型 / VAD / 去噪 / 语言模型）。本课的 `wer()` 与 `alignment()` 错误统计直接对应 `aurora.speech.metrics` 中 `wer` / `corpus_wer` 的核心逻辑（实际路径：`src/aurora/speech/metrics.py`）。

注意：这里的 WER 是**语料级 WER**（把所有句子的 S/D/I 与 N_ref 先累加，再做一次总除法），不是先算每句 WER 再平均。语料级口径更接近 ASR benchmark 的常用评测方式，因为长句会按参考词数自动加权。

下一课（L75）将把这条错误链画出来：波形 → 声谱图 → token 路径 → 文字，让 WER 中每一种错误都能在图上找到对应的时频位置。

In [ ]:
# ── 独立数学断言：alignment 核心性质（使用参考实现，无需学生代码）────────────────

# 1. 完全匹配 → 全M操作，edit_distance=0
ops_id = alignment("a b c", "a b c")
assert all(op == 'M' for op, *_ in ops_id), "完全匹配应全为M操作"
edit_id = sum(1 for op, *_ in ops_id if op != 'M')
assert edit_id == 0, f"完全匹配编辑距离应=0，得{edit_id}"
print(f"1 ✅  identity: alignment('a b c','a b c')→全M，编辑距离=0")

# 2. 单词替换 → S=1
ops_s = alignment("a b c", "a x c")
S2 = sum(1 for op, *_ in ops_s if op == 'S')
assert S2 == 1, f"'b→x'应为1次替换，得S={S2}"
print(f"2 ✅  substitution: 'b→x'，S={S2}")

# 3. 单词删除 → D=1
ops_d = alignment("a b c", "a c")
D3 = sum(1 for op, *_ in ops_d if op == 'D')
assert D3 == 1, f"删除'b'应为D=1，得D={D3}"
print(f"3 ✅  deletion: 删除'b'，D={D3}")

# 4. 单词插入 → I=1
ops_i = alignment("a c", "a b c")
I4 = sum(1 for op, *_ in ops_i if op == 'I')
assert I4 == 1, f"插入'b'应为I=1，得I={I4}"
print(f"4 ✅  insertion: 插入'b'，I={I4}")

# 5. WER公式验证：ref="a b c d"，hyp="x b c"→S=1('a→x'),D=1('d'),I=0,N=4→WER=0.5
ops_wer = alignment("a b c d", "x b c")
S5 = sum(1 for op, *_ in ops_wer if op == 'S')
D5 = sum(1 for op, *_ in ops_wer if op == 'D')
I5 = sum(1 for op, *_ in ops_wer if op == 'I')
wer5 = (S5 + D5 + I5) / 4
assert abs(wer5 - 0.5) < 1e-9, f"WER应=0.5，得{wer5:.4f}"
print(f"5 ✅  WER公式: S={S5},D={D5},I={I5},N=4 → WER={wer5:.2f}")


In [ ]:
# ✏️ 本课自评
l74_review = {
    "wer_impl":              None,  # wer()用alignment()统计S/D/I实现正确，两组验证通过？True/False
    "sdi_three_types":       None,  # 能区分S/D/I三类错误，并说出每类的典型成因（形近词/句末低能量/噪声）？True/False
    "counter_analysis":      None,  # 会用Counter把错误溯源到具体词汇（最常见替换/删除/插入）？True/False
    "error_fix_mapping":     None,  # 记住S/D/I主导时各自的优先改进方向（LM/VAD/降噪）？True/False
    "sdi_self_consistent":   None,  # 理解S+D+I=总编辑操作数，WER分解自洽验证通过？True/False
}

unfilled = [k for k, v in l74_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l74_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L74 全部通关！进入 L75：ASR 流水线图形化')

---

→ **下一课**　[L75 · ASR 流水线图形化](L75_visual_asr.ipynb)

> 下节课将学习 **ASR 流水线图形化**：波形 → 声谱图 → token → 文字路径可视化。